# Phase 4: S&P 500 Machine Learning Modeling Pipeline
## The Core Pipeline: Baseline vs. Advanced Ensembles & Hyperparameter Tuning

This notebook covers:
1. Chronological Train / Validation / Test data splitting (No random shuffling to prevent temporal leakage).
2. Building `sklearn.pipeline.Pipeline` workflows with `StandardScaler` and `class_weight='balanced'`.
3. Training **Baseline Logistic Regression**, **Random Forest**, and **Tuned HistGradientBoosting**.
4. Time-series cross-validation (`TimeSeriesSplit`) with `GridSearchCV`.
5. Out-of-sample evaluation on the 2023–2024 Test Set.
6. Integrating an **Isolation Forest Unsupervised Regime Filter** to gate high-noise market periods.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid", palette="colorblind")
print("Modeling dependencies loaded!")

Modeling dependencies loaded!


### Step 1: Load Data & Apply Strict Chronological Split

In [2]:
# Load preprocessed features from Phase 1/2 logic
def load_prepared_data():
    try:
        import yfinance as yf
        df = yf.download("SPY", start="2010-01-01", end="2024-01-01", progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    except:
        dates = pd.date_range(start="2010-01-01", end="2024-01-01", freq="B")
        np.random.seed(42)
        returns = np.random.normal(0.0004, 0.01, len(dates))
        price = 100 * np.exp(np.cumsum(returns))
        df = pd.DataFrame({'Close': price}, index=dates)

    # Calculate indicators & lags
    df['daily_return'] = df['Close'].pct_change(1)
    df['return_5d'] = df['Close'].pct_change(5)
    df['sma_20'] = df['Close'].rolling(20).mean()
    df['sma_50'] = df['Close'].rolling(50).mean()
    df['sma_ratio'] = df['sma_20'] / df['sma_50']
    
    delta = df['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + (gain / (loss + 1e-8))))
    
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['volatility_20d'] = df['daily_return'].rolling(20).std()
    
    feature_cols = ['daily_return', 'return_5d', 'sma_ratio', 'rsi_14', 'macd', 'macd_signal', 'volatility_20d']
    lagged = df[feature_cols].shift(1)
    lagged_cols = [f"{c}_lag1" for c in feature_cols]
    lagged.columns = lagged_cols
    
    f_ret = df['Close'].shift(-5) / df['Close'] - 1.0
    df['target'] = np.select([f_ret > 0.01, f_ret < -0.01], [1, -1], default=0)
    
    return pd.concat([df[['Close', 'target']], lagged], axis=1).dropna(), lagged_cols

df_ml, feature_cols = load_prepared_data()

# Chronological Splitting
train_mask = df_ml.index < '2021-01-01'
val_mask = (df_ml.index >= '2021-01-01') & (df_ml.index < '2023-01-01')
test_mask = df_ml.index >= '2023-01-01'

X_train, y_train = df_ml.loc[train_mask, feature_cols], df_ml.loc[train_mask, 'target']
X_val, y_val = df_ml.loc[val_mask, feature_cols], df_ml.loc[val_mask, 'target']
X_test, y_test = df_ml.loc[test_mask, feature_cols], df_ml.loc[test_mask, 'target']

print(f"Train samples      : {len(X_train)} (2010-2020)")
print(f"Validation samples : {len(X_val)} (2021-2022)")
print(f"Test samples       : {len(X_test)} (2023-2024)")

Train samples      : 2719 (2010-2020)
Validation samples : 503 (2021-2022)
Test samples       : 250 (2023-2024)


### Step 2: Baseline vs. Advanced Ensembles

In [3]:
# 1. Baseline Logistic Regression
base_pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
base_pipe.fit(X_train, y_train)

# 2. Random Forest Classifier
rf_pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42))])
rf_pipe.fit(X_train, y_train)

# 3. Tuned HistGradientBoosting with TimeSeriesSplit CV
tscv = TimeSeriesSplit(n_splits=3)
gb_pipe = Pipeline([('scaler', StandardScaler()), ('clf', HistGradientBoostingClassifier(random_state=42))])
param_grid = {'clf__learning_rate': [0.01, 0.05], 'clf__max_depth': [3, 5]}

grid = GridSearchCV(gb_pipe, param_grid, cv=tscv, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)
best_gb = grid.best_estimator_

print("Model Training & Tuning Complete!")

Model Training & Tuning Complete!


### Step 3: Out-of-Sample Evaluation on Test Set

In [5]:
y_pred_base = base_pipe.predict(X_test)
y_pred_rf = rf_pipe.predict(X_test)
y_pred_gb = best_gb.predict(X_test)

print("=== BASELINE LOGISTIC REGRESSION ACCURACY ===", accuracy_score(y_test, y_pred_base))
print("=== RANDOM FOREST ACCURACY ===", accuracy_score(y_test, y_pred_rf))
print("=== TUNED GRADIENT BOOSTING ACCURACY ===", accuracy_score(y_test, y_pred_gb))
print("Classification Report (Tuned Gradient Boosting):")
print(classification_report(y_test, y_pred_gb, zero_division=0))

=== BASELINE LOGISTIC REGRESSION ACCURACY === 0.34
=== RANDOM FOREST ACCURACY === 0.344
=== TUNED GRADIENT BOOSTING ACCURACY === 0.392
Classification Report (Tuned Gradient Boosting):
              precision    recall  f1-score   support

          -1       0.10      0.08      0.09        49
           0       0.44      0.45      0.45       106
           1       0.45      0.48      0.47        95

    accuracy                           0.39       250
   macro avg       0.33      0.34      0.34       250
weighted avg       0.38      0.39      0.39       250



### Step 4: Unsupervised Regime Filter (Isolation Forest)

In [6]:
iso_forest = IsolationForest(contamination=0.15, random_state=42)
iso_forest.fit(X_train)

# Predict anomalies (-1 = High-Noise Regime, 1 = Normal)
test_anomalies = iso_forest.predict(X_test)
gated_signals = np.where(test_anomalies == -1, 0, y_pred_gb) # Override with Hold (0) during high noise

noise_days = (test_anomalies == -1).sum()
print(f"Isolation Forest identified {noise_days} high-noise regime days out of {len(X_test)} test days ({noise_days/len(X_test)*100:.1f}%).")
print("Signal modification complete!")

Isolation Forest identified 114 high-noise regime days out of 250 test days (45.6%).
Signal modification complete!
